# Day 4 — Safety, Guardrails & Internal Evaluation
### MedFlow · Evidence-Grounded Medical RAG for Thyroid Diseases

**Goal:** move from cited answers to measurably safe answers.

Day 4 measures four things: threshold behavior, unsupported claims, citation accuracy, and faithfulness. The notebook is deliberately non-destructive: it audits the existing index before any full evaluation.

## 0. Frozen Day 2 contract

The evaluation must keep the Day 2 retriever fixed: **BGE-small-en-v1.5 + 200 tokens + 0 overlap + Top-K=4 + no reranker**. We do not retune retrieval just to improve Day 4 numbers.

In [ ]:
from day4 import config
print("Embedding:", config.EMBEDDING_MODEL_NAME)
print("Chunking:", config.CHUNK_SIZE_TOKENS, "tokens /", config.CHUNK_OVERLAP_TOKENS, "overlap")
print("Top-K:", config.TOP_K)
print("Expected indexed chunks:", config.EXPECTED_INDEXED_CHUNKS)

## 1. Audit the persisted index

This check is intentionally read-only. A frozen benchmark is only defensible if the evaluated index matches the frozen configuration.

In [ ]:
from day4.index_audit import audit_index
audit = audit_index()
audit

If `index_matches_frozen_day2` is `False`, do not present Day 4 metrics as frozen-Day-2-comparable. Build the separate exact index with:

```bash
python day4/frozen_index_builder.py
```

## 2. Threshold calibration concept

We label known Day 2 queries as answerable and Day 3 refusal cases as unsupported, collect their top retrieval scores, sweep thresholds, and choose a safety-constrained operating point. The threshold is therefore measured rather than copied from a slide.

In [ ]:
from day4.threshold_calibration import calibrate_threshold

# Small transparent example; the real evaluator uses the project's 16 + 10 cases.
demo_samples = [
    {"expected_answerable": True, "top_score": 0.82},
    {"expected_answerable": True, "top_score": 0.75},
    {"expected_answerable": True, "top_score": 0.72},
    {"expected_answerable": False, "top_score": 0.42},
    {"expected_answerable": False, "top_score": 0.36},
    {"expected_answerable": False, "top_score": 0.30},
]
calibrate_threshold(demo_samples, max_unsafe_accept_rate=0.0)["selected_metrics"]

## 3. Unsupported-claim detection

The post-hoc guard splits the recommendation into factual claims and checks each claim against retrieved evidence. Numerical and dosage claims are conservative: their numbers and units must occur in a supporting passage.

In [ ]:
from day4.claim_validator import evaluate_faithfulness

answer = "Propylthiouracil should be given at 5 mg/kg/day."
evidence = [{"retrieved_passage": "Antithyroid drugs include methimazole and propylthiouracil."}]
evaluate_faithfulness(answer, evidence, lexical_threshold=0.20)

## 4. The three named Day 4 metrics

- **Precision@4** = relevant retrieved chunks / 4
- **Citation Accuracy** = correct citations / citations given
- **Faithfulness** = supported claims / claims made

Target faithfulness is **>= 0.90**.

In [ ]:
from day4 import config
print("Faithfulness target:", config.TARGET_FAITHFULNESS)
print("Minimum citation accuracy for live guard:", config.MIN_CITATION_ACCURACY)

## 5. Responsible AI

The guard does not convert weak evidence into confident prose. Below threshold it refuses. Weak or partial evidence is described as weak or partial, and a visible clinical disclaimer is carried in the safety diagnostics.

In [ ]:
from day4.safety_guardrails import uncertainty_language
from day4.config import CLINICAL_DISCLAIMER
print(uncertainty_language(0.50, 0.65, 1.0, 1.0))
print("\nDisclaimer:", CLINICAL_DISCLAIMER)

## 6. Run the real internal evaluation

After building/auditing the exact frozen index, run from the project root:

```bash
python day4/evaluate_day4.py --full --persist-dir chroma_db_day2_frozen --collection thyroid_day2_frozen
```

The evaluator writes the threshold sweep, per-query log, and final summary into `results/day4/`. Those are the numbers to present in Day 5 — not the illustrative notebook values.